In [0]:
# Mosaic AI exploration
## - MLflow setup
import os, mlflow

MLRUNS_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/mlruns"
os.makedirs(MLRUNS_PATH, exist_ok=True)

mlflow.set_tracking_uri(f"file:{MLRUNS_PATH}")
mlflow.set_experiment("day14_genai_nlp")

print("tracking_uri:", mlflow.get_tracking_uri())


In [0]:
# Sentiment pipeline with fallback + run logging

import time
import mlflow
import pandas as pd

reviews = [
    "This product is amazing, battery life is insane.",
    "Terrible quality. Stopped working in two days.",
    "Good value for money, not perfect but solid.",
    "Waste of money. I want a refund.",
    "Surprisingly good build quality for the price."
]

def rule_based_sentiment(text: str) -> dict:
    t = text.lower()
    pos = ["amazing", "great", "good", "love", "solid", "insane", "excellent"]
    neg = ["terrible", "waste", "refund", "broken", "bad", "awful", "stopped working"]
    score = sum(w in t for w in pos) - sum(w in t for w in neg)
    label = "POSITIVE" if score >= 0 else "NEGATIVE"
    return {"label": label, "score": float(abs(score))}

with mlflow.start_run(run_name="sentiment_inference"):
    mlflow.log_param("task", "sentiment-analysis")

    t0 = time.time()
    try:
        from transformers import pipeline
        clf = pipeline("sentiment-analysis")  # default model (usually SST-2 fine-tuned DistilBERT)
        preds = clf(reviews)
        mlflow.log_param("backend", "transformers.pipeline")
    except Exception as e:
        preds = [rule_based_sentiment(r) for r in reviews]
        mlflow.log_param("backend", "rule_based_fallback")
        mlflow.log_param("fallback_reason", str(e)[:200])

    latency_s = time.time() - t0
    mlflow.log_metric("inference_latency_sec", float(latency_s))
    mlflow.log_metric("n_reviews", float(len(reviews)))

    out = pd.DataFrame({
        "review": reviews,
        "label": [p["label"] for p in preds],
        "score": [float(p["score"]) for p in preds],
    })

    # Persist results as artifact
    out_path = "/tmp/day14_sentiment_results.csv"
    out.to_csv(out_path, index=False)
    mlflow.log_artifact(out_path, artifact_path="outputs")

out

In [0]:
# AI-assisted analysis: generate a narrative insight from your KPI tables
import mlflow
import pandas as pd
from pyspark.sql import functions as F

kpis = spark.table("gold.daily_kpis")

# Pick the last 14 days (adjust column name if needed)
date_col = "event_date" if "event_date" in kpis.columns else kpis.columns[0]

recent = (
    kpis
    .withColumn("d", F.to_date(F.col(date_col)))
    .orderBy(F.col("d").desc())
    .limit(14)
)

pdf = recent.toPandas()
pdf = pdf.sort_values("d")

# Heuristic insight: day-over-day change for purchases/revenue if present
def pick_col(cands):
    for c in cands:
        if c in pdf.columns:
            return c
    return None

purchases_col = pick_col(["purchases", "purchase_count"])
revenue_col = pick_col(["revenue", "total_revenue"])

insights = []
if purchases_col:
    pdf["purch_dod"] = pdf[purchases_col].diff()
    biggest_up = pdf.loc[pdf["purch_dod"].idxmax()]
    biggest_down = pdf.loc[pdf["purch_dod"].idxmin()]
    insights.append(f"Biggest purchase lift: {biggest_up['d']} (+{biggest_up['purch_dod']:.0f}).")
    insights.append(f"Biggest purchase drop: {biggest_down['d']} ({biggest_down['purch_dod']:.0f}).")

if revenue_col:
    pdf["rev_dod"] = pdf[revenue_col].diff()
    r_up = pdf.loc[pdf["rev_dod"].idxmax()]
    r_down = pdf.loc[pdf["rev_dod"].idxmin()]
    insights.append(f"Biggest revenue lift: {r_up['d']} (+{r_up['rev_dod']:.2f}).")
    insights.append(f"Biggest revenue drop: {r_down['d']} ({r_down['rev_dod']:.2f}).")

if not insights:
    insights.append("daily_kpis did not contain expected metric columns; adjust column mapping and rerun.")

insight_text = "\n".join(insights)
insight_text


In [0]:
# Logging the insight as an MLflow artifact
import mlflow

with mlflow.start_run(run_name="ai_assisted_insights"):
    mlflow.log_param("source_table", "gold.daily_kpis")
    path = "/tmp/day14_insights.txt"
    with open(path, "w") as f:
        f.write(insight_text)
    mlflow.log_artifact(path, artifact_path="insights")

print(insight_text)
